# 06 — Tích hợp kết quả Validation (Người 1)


## Mở project
**Colab:** đặt đầy đủ thư mục project tại `MyDrive/student-dropout-prediction`,
gồm notebook, `common/`, `requirements.txt` và `data/raw/data.csv`.
Nếu thư mục có tên khác, sửa `PROJECT_ROOT` ở cell dưới. Mở mỗi notebook rồi chạy từ trên xuống.
**VS Code/Jupyter:** mở notebook từ thư mục `notebooks/`; cell tự tìm thư mục gốc.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/student-dropout-prediction')
else:
    here = Path.cwd().resolve()
    PROJECT_ROOT = next((p for p in [here, *here.parents]
                         if (p / 'common' / 'data_io.ipynb').is_file()), here)

if not (PROJECT_ROOT / 'common' / 'data_io.ipynb').is_file():
    raise FileNotFoundError('Sửa PROJECT_ROOT tới thư mục chứa common/ và requirements.txt.')
os.chdir(PROJECT_ROOT)
print('Project:', PROJECT_ROOT)


## Thư viện
Colab không cần `venv`, nhưng vẫn cần đúng thư viện. Cell này đọc `requirements.txt`.
Nếu Colab yêu cầu khởi động lại runtime sau khi cài, thực hiện rồi chạy lại từ đầu.


In [ ]:
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])


In [ ]:
import json
import pandas as pd
from IPython.display import display
%run common/data_io.ipynb
%run common/splits.ipynb

DATA_PATH = PROJECT_ROOT / 'data/raw/data.csv'
SPLIT_PATH = PROJECT_ROOT / 'data/splits/splits.json'
ARTIFACTS = PROJECT_ROOT / 'artifacts'
TABLES = PROJECT_ROOT / 'reports/tables'
FIGURES = PROJECT_ROOT / 'reports/figures'
for folder in [SPLIT_PATH.parent, ARTIFACTS, TABLES, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Đặt CSV gốc tại {DATA_PATH}')


In [ ]:
available_runs = sorted(p.parent for p in ARTIFACTS.glob('*/*/config.json'))
for folder in available_runs:
    print(folder.relative_to(PROJECT_ROOT))


Chọn rõ lượt chạy đã thống nhất của từng mô hình. Điền đường dẫn in ở cell trên vào danh sách; không tự chọn lượt mới nhất.


In [ ]:
SELECTED_RUNS = [
    # 'artifacts/dummy/20260101T000000Z_abcd1234',
    # 'artifacts/logistic/20260101T000000Z_abcd1234',
]


In [ ]:
%run common/training.ipynb
if SELECTED_RUNS:
    comparison = compare_validation([PROJECT_ROOT / p for p in SELECTED_RUNS], SPLIT_PATH,
                                    TABLES / 'validation_comparison.csv')
    display(comparison)
else:
    print('Điền SELECTED_RUNS để tạo bảng so sánh.')
